# 🧵 template 深掘り

tools / vision / reasoning_effort / 長文トークン効率。


In [ ]:
import json
import os
import subprocess
import sys


def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    # slug 一致を最優先。無ければ qwen を含むもの、最後に全候補。
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]


ROOT = find_root()

print("transformers を用意")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                "transformers==5.16.1"], capture_output=True, text=True)

from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(ROOT)
print("tokenizer:", type(tok).__name__)
print("chat_template 解決:", tok.chat_template is not None,
      len(tok.chat_template or ""))
print("eos:", tok.eos_token, tok.eos_token_id,
      "| bos:", tok.bos_token, tok.bos_token_id,
      "| pad:", tok.pad_token)

MSGS = [{"role": "system", "content": "You are concise."},
        {"role": "user", "content": "1+1 は？"}]

TOOLS = [{"type": "function", "function": {
    "name": "get_weather",
    "description": "天気を返す",
    "parameters": {"type": "object",
                   "properties": {"city": {"type": "string"}},
                   "required": ["city"]}}}]


def render(label, **kw):
    try:
        s = tok.apply_chat_template(MSGS, tokenize=False,
                                    add_generation_prompt=True, **kw)
    except Exception as e:
        print(f"\n--- {label}: ERROR {type(e).__name__}: {e}")
        return
    ids = tok.encode(s, add_special_tokens=False)
    print(f"\n--- {label}: chars={len(s)} tokens={len(ids)}")
    print("   head:", repr(s[:200]))
    print("   tail:", repr(s[-160:]))


render("basic")
for eff in ("low", "medium", "high", "xhigh"):
    render(f"reasoning_effort={eff}", reasoning_effort=eff)
render("tools", tools=TOOLS)
render("tools+eff=low", tools=TOOLS, reasoning_effort="low")

print("\n=== tool_call の XML 形式 ===")
s = tok.apply_chat_template(MSGS + [{"role": "assistant", "content":
                                    '<tool_call><function=get_weather>'
                                    '<parameter=city>Tokyo</parameter>'
                                    '</function></tool_call>'}],
                            tokenize=False)
print(repr(s[-260:]))
print("tokenize 往復:", tok.decode(tok.encode(s, add_special_tokens=False))[:80])

print("\n=== vision content ===")
vis = [{"role": "user", "content": [
    {"type": "image", "image": "x.png"},  # ダミー。レンダラだけ確認
    {"type": "text", "text": "これは？"}]}]
try:
    s = tok.apply_chat_template(vis, tokenize=False, add_generation_prompt=True)
    print("   ok chars:", len(s))
    print("   ", repr(s[:200]))
except Exception as e:
    print("   ERROR:", type(e).__name__, e)

print("\n=== 長文でのトークン効率（4k 文字）===")
long_msgs = [{"role": "user", "content": "あ" * 4000}]
s = tok.apply_chat_template(long_msgs, tokenize=False, add_generation_prompt=True)
print("   chars:", len(s), "tokens:", len(tok.encode(s, add_special_tokens=False)))

print("\n=== generation_config / 特殊トークン ===")
for name in ("generation_config.json", "special_tokens_map.json"):
    p = os.path.join(ROOT, name)
    print(f"  {name}: {'あり' if os.path.isfile(p) else 'なし'}")
    if os.path.isfile(p):
        print("   ", open(p, encoding="utf-8").read()[:400])
